# Custom Tools Agent

In this notebook we build a general restaurant-style agent with custom tools.

Custom tools are normal Python functions that we give to the LLM.

The LLM can call these tools when it needs real actions or real information.

Tools in this notebook:

- `check_menu`
- `check_order_status`
- `calculate_bill`
- `get_weather`
- `web_search`
- `get_current_datetime`

## Why Custom Tools?

An LLM can talk, explain, and reason.

But by itself, it cannot check your restaurant menu, calculate exact bill from your prices, check order status, or get live weather.

Custom tools solve this.

Easy example:

A waiter may be smart, but the waiter still checks the menu card, bill machine, order screen, and weather app.

In an agent:

| Real life | Agent |
| --- | --- |
| Menu card | `check_menu` tool |
| Order screen | `check_order_status` tool |
| Bill machine | `calculate_bill` tool |
| Weather app | `get_weather` tool |
| Internet browser | `web_search` tool |
| Clock/calendar | `get_current_datetime` tool |

## Step 1: Imports

We import Python libraries, LangChain tools, Groq LLM, LangGraph, and Gradio.

`@tool` converts a Python function into a tool the LLM can call.

`ToolNode` runs the selected tool inside LangGraph.

In [25]:
import json
import os
import re
import urllib.parse
import urllib.request
from datetime import datetime
from zoneinfo import ZoneInfo

import gradio as gr
from dotenv import load_dotenv
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

## Step 2: Load Environment

We load `.env` so the notebook can read `GROQ_API_KEY`.

We also turn off LangSmith tracing for learning, so you do not see auth warnings.

In [26]:
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGCHAIN_TRACING"] = "false"

print("GROQ_API_KEY:", "set" if os.getenv("GROQ_API_KEY") else "missing")

GROQ_API_KEY: set


## Step 3: Custom Tool 1 - Check Menu

Purpose: check if a food item exists in our menu.

Benefit: the LLM does not guess prices. It reads our actual menu dictionary.

Daily life example: the waiter checks the menu card before answering.

In [27]:
MENU = {
    "biryani": 450,
    "burger": 300,
    "pizza": 1200,
    "karahi": 1800,
    "fries": 200,
    "tea": 120,
}


@tool
def check_menu(item: str) -> str:
    """Check if a food item is available in the restaurant menu and return its price."""
    item = item.lower().strip()

    if item in MENU:
        return f"{item.title()} is available. Price is Rs {MENU[item]}."

    available_items = ", ".join(MENU.keys())
    return f"{item.title()} is not available. Available items are: {available_items}."

In [28]:
check_menu.invoke({"item": "biryani"})

'Biryani is available. Price is Rs 450.'

## Step 4: Custom Tool 2 - Check Order Status

Purpose: check an order using an order ID.

Benefit: the agent can answer from real order data instead of making up a status.

Daily life example: restaurant staff checks the order screen.

In [29]:
ORDERS = {
    "101": "Your biryani is being cooked and will be ready in 10 minutes.",
    "102": "Your pizza is out for delivery.",
    "103": "Your refund request is processing.",
}


@tool
def check_order_status(order_id: str) -> str:
    """Check the status of a restaurant order by order ID."""
    order_id = order_id.strip()
    return ORDERS.get(order_id, "Order ID not found. Please check the order number.")

In [30]:
check_order_status.invoke({"order_id": "101"})

'Your biryani is being cooked and will be ready in 10 minutes.'

## Step 5: Custom Tool 3 - Calculate Bill

Purpose: calculate total bill from item names and quantities.

Benefit: exact math. The LLM does not make calculation mistakes.

Input format is JSON text, for example:

```json
{"biryani": 2, "tea": 3}
```

In [31]:
@tool
def calculate_bill(order_json: str) -> str:
    """Calculate a restaurant bill. Input must be JSON like {\"biryani\": 2, \"tea\": 3}."""
    try:
        order = json.loads(order_json)
    except Exception:
        return "Invalid JSON. Example: {\"biryani\": 2, \"tea\": 3}"

    total = 0
    lines = []

    for item, quantity in order.items():
        item = item.lower().strip()
        if item not in MENU:
            lines.append(f"{item.title()} is not in the menu.")
            continue

        price = MENU[item]
        subtotal = price * int(quantity)
        total += subtotal
        lines.append(f"{item.title()} x {quantity} = Rs {subtotal}")

    lines.append(f"Total bill = Rs {total}")
    return "\n".join(lines)

In [32]:
print(calculate_bill.invoke({"order_json": "{\"biryani\": 2, \"tea\": 3}"}))

Biryani x 2 = Rs 900
Tea x 3 = Rs 360
Total bill = Rs 1260


## Step 6: Custom Tool 4 - Current Date And Time

Purpose: get current date and time.

Benefit: the model does not guess today's date.

Daily life example: checking a clock/calendar.

In [33]:
@tool
def get_current_datetime(timezone: str = "Asia/Karachi") -> str:
    """Get the current date and time for a timezone."""
    try:
        now = datetime.now(ZoneInfo(timezone))
    except Exception:
        now = datetime.now()

    return now.strftime("%A, %B %d, %Y %I:%M %p")

In [34]:
get_current_datetime.invoke({"timezone": "Asia/Karachi"})

'Tuesday, July 07, 2026 10:11 AM'

## Step 7: Custom Tool 5 - Weather

Purpose: get current weather for a city.

Benefit: weather changes all the time, so the LLM needs a live tool.

This uses Open-Meteo, which is free-friendly and does not need an API key.

In [35]:
@tool
def get_weather(city: str) -> str:
    """Get current weather for a city using Open-Meteo."""
    city = city.strip()
    if not city:
        return "Please provide a city name."

    geocode_url = "https://geocoding-api.open-meteo.com/v1/search?" + urllib.parse.urlencode({
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json"
    })

    with urllib.request.urlopen(geocode_url, timeout=15) as response:
        geocode_data = json.loads(response.read().decode("utf-8"))

    results = geocode_data.get("results", [])
    if not results:
        return f"I could not find weather coordinates for {city}."

    place = results[0]
    latitude = place["latitude"]
    longitude = place["longitude"]
    place_name = place.get("name", city)
    country = place.get("country", "")

    weather_url = "https://api.open-meteo.com/v1/forecast?" + urllib.parse.urlencode({
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m,relative_humidity_2m,wind_speed_10m",
        "timezone": "auto"
    })

    with urllib.request.urlopen(weather_url, timeout=15) as response:
        weather_data = json.loads(response.read().decode("utf-8"))

    current = weather_data.get("current", {})
    temperature = current.get("temperature_2m")
    humidity = current.get("relative_humidity_2m")
    wind = current.get("wind_speed_10m")

    return f"Current weather in {place_name}, {country}: temperature {temperature} C, humidity {humidity}%, wind speed {wind} km/h."

In [36]:
get_weather.invoke({"city": "Karachi"})

'Current weather in Karachi, Pakistan: temperature 32.1 C, humidity 70%, wind speed 9.3 km/h.'

## Step 8: Custom Tool 6 - Web Search

Purpose: search the web for latest/current information.

Benefit: the LLM can get information that may not exist in its training data.

This uses LangChain's DuckDuckGo search integration.

In [37]:
duckduckgo_search = DuckDuckGoSearchRun()


@tool
def web_search(query: str) -> str:
    """Search the web for current or latest information."""
    query = query.strip()
    if not query:
        return "Please provide a search query."

    try:
        return duckduckgo_search.invoke(query)
    except Exception as error:
        return f"Search tool error: {error}. Try again or use a different query."

In [38]:
web_search.invoke({"query": "latest AI news today"})

"At Google I/O 2026, it’s AI, AI, and more AI. 2 hours ago. By Andrew Nusca. Save for later. Share.Rolling out today starting with video outputs to Google AI Plus, pic.x.com/EkLjv5O0dN. AI news from every angle. Curated AI news, research, and tools — no hype, no filler. Just the stories that matter. Today's Briefing. Latest. Following. Hot. Get the latest news and stories about Google products, technology and innovation on the Keyword, Google's official blog.ISTE 2026: Supporting teaching and learning with connected AI tools. The latest announcements from Google for Education at ISTE 2026. Stay up to date with the latest AI news from OpenAI, Anthropic, Google DeepMind, Meta AI, and more. Curated stories from 30+ sources, updated every 15 minutes. Tulsee Doshi, Head of Product for Gemini Models joins host Logan Kilpatrick for an in-depth discussion on the latest Gemini 2.5 Pro experimental launch. Gemini 2.5 is a well-rounded, multimodal thinking model, designed to tackle increasingly c

## Step 9: Give Tools To The LLM

Now we collect all tools into one list.

`bind_tools` gives the LLM permission to call these tools.

The model decides when a tool is needed.

In [39]:
tools = [
    check_menu,
    check_order_status,
    calculate_bill,
    get_current_datetime,
    get_weather,
    web_search,
]

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
llm_with_tools = llm.bind_tools(tools)

## Step 10: Build LangGraph Agent

The graph has two nodes:

1. `assistant`: the LLM thinks and may request a tool
2. `tools`: LangGraph runs the requested tool

If the LLM calls a tool, the graph goes to `tools` and then back to `assistant`.

If the LLM does not call a tool, the graph ends.

In [41]:
system_prompt = """
You are a helpful restaurant and general assistant.
Use tools whenever they are useful.
Use check_menu for menu and price questions.
Use check_order_status for order tracking.
Use calculate_bill for bill calculations.
Use get_current_datetime for date/time questions.
Use get_weather for weather questions.
Use web_search for latest/current information.
Keep answers simple and beginner friendly.
"""


def assistant_node(state: MessagesState):
    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


tool_node = ToolNode(tools)

builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant_node)
builder.add_node("tools", tool_node)
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")

custom_tool_agent = builder.compile()

## Step 11: Test The Agent

Try different questions.

The agent should choose the right tool.

In [42]:
def ask_agent(question: str) -> str:
    response = custom_tool_agent.invoke({"messages": [HumanMessage(content=question)]})
    return response["messages"][-1].content

In [43]:
ask_agent("Is biryani available and what is the price?")

'The price of biryani is Rs 450.'

In [44]:
ask_agent("Check order status for 101")

'Your order with ID 101 is being prepared and will be ready soon.'

In [45]:
ask_agent('Calculate bill for {"biryani": 2, "tea": 3}')

'Your total bill is Rs 1260.'

In [46]:
ask_agent("What is the current weather in Karachi?")

'The current weather in Karachi is 32.1 degrees Celsius with a humidity of 70% and a wind speed of 9.3 km/h.'

In [47]:
ask_agent("Search latest AI news today")

'The latest AI news today includes a massive new study that found generative AI can now beat the average human on certain creativity tests, and updates on the latest advancements in artificial intelligence, machine learning, and deep learning. Additionally, there are articles on the ethical issues AI raises and the companies building AI technology.'

## Step 12: Gradio App

Now we make a small app.

Type a question like:

- Is pizza available?
- Check order status 102
- Calculate bill for two biryani and one tea
- What is the weather in Karachi?
- Search latest AI news today

In [48]:
def custom_tools_app(question: str) -> str:
    if not question.strip():
        return "Please type a question."

    try:
        return ask_agent(question)
    except Exception as error:
        return f"Error: {error}"


demo = gr.Interface(
    fn=custom_tools_app,
    inputs=gr.Textbox(label="Ask the Agent", placeholder="Example: Is biryani available?"),
    outputs=gr.Textbox(label="Agent Answer"),
    title="Custom Tools Agent",
    description="Agent with menu, order status, bill, weather, search, and date/time tools."
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## Benefits Of Custom Tools

Custom tools make an agent more useful because the agent can do real work.

Benefits:

1. More accurate answers because tools use real data
2. Less guessing by the LLM
3. Live information like weather and search
4. Business logic like menu, order status, and billing
5. Easy to extend later with database, email, calendar, camera, or payment tools

In simple words: tools turn a chatbot into an assistant that can actually do tasks.